# SentimentStream - Fase 2: Exploración del dataset

## Propósito

Este notebook realiza una exploración inicial y conservadora del archivo `data/raw/dataset_sentimientos_500.csv`.

El objetivo es diagnosticar la estructura general del dataset, revisar calidad básica de datos y documentar hallazgos relevantes para fases posteriores.

**Alcance:** esta fase no entrena modelos, no realiza limpieza avanzada, no ejecuta pipelines PySpark, no persiste información en bases de datos y no implementa API, Docker ni Jenkins.

## 1. Importación de librerías

Se usan únicamente librerías necesarias para exploración tabular y visualización básica.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Definición de ruta y validación del archivo

Antes de analizar datos, se valida que el archivo exista en la ruta esperada del proyecto.

In [ ]:
current_dir = Path.cwd().resolve()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw").exists() and (path / "spark" / "notebooks").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "No se pudo identificar la raíz del proyecto SentimentStream desde el directorio actual."
    )

dataset_path = project_root / "data" / "raw" / "dataset_sentimientos_500.csv"

if not dataset_path.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset esperado en: {dataset_path}. "
        "Ubique el archivo antes de continuar con la exploración."
    )

print(f"Raíz del proyecto: {project_root}")
print(f"Dataset detectado: {dataset_path}")

## 3. Carga del dataset

Se carga el archivo CSV con pandas y se inspeccionan las primeras filas, dimensiones y columnas disponibles.

In [ ]:
df = pd.read_csv(dataset_path)

display(df.head())
print(f"Dimensiones del dataset: {df.shape[0]} registros y {df.shape[1]} columnas")
print("Columnas disponibles:")
for column in df.columns:
    print(f"- {column}")

## 4. Estructura general

Se revisan tipos de datos, estructura general y valores nulos por columna.

In [ ]:
df.info()

estructura = pd.DataFrame(
    {
        "tipo_dato": df.dtypes.astype(str),
        "valores_nulos": df.isna().sum(),
        "porcentaje_nulos": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=False),
    }
)

estructura

## 5. Revisión de duplicados

Se identifican duplicados completos. Si existe una columna `id`, también se revisa si parece comportarse como identificador único.

In [ ]:
duplicados_generales = int(df.duplicated().sum())
porcentaje_duplicados = round((duplicados_generales / len(df)) * 100, 2) if len(df) else 0

print(f"Registros duplicados completos: {duplicados_generales}")
print(f"Porcentaje de duplicados completos: {porcentaje_duplicados}%")

if "id" in df.columns:
    duplicados_id = int(df["id"].duplicated().sum())
    print(f"Valores duplicados en columna id: {duplicados_id}")
    if duplicados_id == 0:
        print("La columna id parece comportarse como identificador único.")
    else:
        print("La columna id no es completamente única; debe revisarse en una fase posterior.")
else:
    print("El dataset no contiene columna id; no se puede evaluar unicidad por identificador.")

## 6. Exploración de la variable objetivo

La especificación del proyecto menciona la variable `sentimiento`. En el archivo actual la columna disponible para la etiqueta de sentimiento se llama `etiqueta`. Para no modificar el dataset, se analiza esa columna como variable objetivo disponible.

In [ ]:
target_column = None
if "sentimiento" in df.columns:
    target_column = "sentimiento"
elif "etiqueta" in df.columns:
    target_column = "etiqueta"

if target_column is None:
    raise KeyError(
        "No se encontró una columna de sentimiento. Se esperaba 'sentimiento' o, según el archivo actual, 'etiqueta'."
    )

conteo_clases = df[target_column].value_counts(dropna=False)
porcentaje_clases = (df[target_column].value_counts(normalize=True, dropna=False) * 100).round(2)

distribucion_objetivo = pd.DataFrame(
    {
        "conteo": conteo_clases,
        "porcentaje": porcentaje_clases,
    }
)

print(f"Columna objetivo analizada: {target_column}")
display(distribucion_objetivo)

ax = conteo_clases.plot(kind="bar", figsize=(7, 4), color="#4c78a8")
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretación académica inicial:** una distribución balanceada entre clases reduce el riesgo de sesgo por desbalance en fases posteriores. Esta observación no implica que el dataset sea suficiente para modelado; solo describe la proporción de etiquetas disponibles.

## 7. Exploración de textos

Se revisan ejemplos, textos vacíos y longitud en caracteres. Esta revisión no aplica limpieza ni transformación final del texto.

In [ ]:
text_column = "texto" if "texto" in df.columns else None

if text_column is None:
    raise KeyError("No se encontró la columna 'texto', necesaria para la exploración textual.")

print("Ejemplos de textos:")
display(df[[text_column, target_column]].head(10))

textos = df[text_column].fillna("").astype(str)
textos_vacios = int(textos.str.strip().eq("").sum())
longitud_texto = textos.str.len()

print(f"Textos vacíos o con solo espacios: {textos_vacios}")
print("Estadísticas de longitud del texto en caracteres:")
display(longitud_texto.describe().to_frame(name="longitud_caracteres"))

df_texto_resumen = df[[text_column, target_column]].copy()
df_texto_resumen["longitud_caracteres"] = longitud_texto
display(df_texto_resumen.sort_values("longitud_caracteres").head(5))
display(df_texto_resumen.sort_values("longitud_caracteres", ascending=False).head(5))

**Comentario sobre calidad inicial del texto:** el dataset actual no presenta textos nulos ni vacíos. Los textos son cortos, lo cual puede ser adecuado para una primera práctica académica, pero limita la riqueza lingüística disponible para fases posteriores.

## 8. Exploración de fechas

Se revisa si existe una columna `fecha`, si puede convertirse a tipo temporal y si aporta información para análisis cronológico.

In [ ]:
if "fecha" in df.columns:
    fechas = pd.to_datetime(df["fecha"], errors="coerce")
    fechas_invalidas = int(fechas.isna().sum())

    print(f"Valores de fecha inválidos o no convertibles: {fechas_invalidas}")
    print(f"Fecha mínima: {fechas.min()}")
    print(f"Fecha máxima: {fechas.max()}")

    conteo_por_fecha = fechas.dt.date.value_counts().sort_index()
    display(conteo_por_fecha.to_frame(name="cantidad_registros").head(20))
else:
    print("El dataset no contiene columna 'fecha'.")
    print("Con el archivo actual no es posible evaluar utilidad temporal ni tendencias por fecha.")

## 9. Hallazgos relevantes para fases posteriores

- El archivo actual contiene 502 registros y 2 columnas: `texto` y `etiqueta`.
- No se observaron valores nulos en las columnas disponibles.
- Se detectaron 472 registros duplicados completos, lo cual debe revisarse con cuidado antes de cualquier fase de limpieza o modelado.
- La variable objetivo disponible es `etiqueta`; no existe una columna llamada `sentimiento` en el CSV actual.
- La distribución de clases es prácticamente balanceada: 168 registros positivos, 167 negativos y 167 neutrales.
- No existe columna `id`, por lo que no se puede validar unicidad por identificador.
- No existe columna `fecha`, por lo que no se puede realizar análisis temporal con este archivo.
- Los textos son cortos y no presentan valores vacíos, según la revisión inicial.

Estos hallazgos son diagnósticos. No constituyen limpieza, transformación final ni selección de variables para modelado.

## 10. Conclusiones

La exploración inicial confirma que el dataset está disponible y puede cargarse correctamente con pandas. La estructura real del archivo es más simple que la estructura esperada inicialmente: contiene `texto` y `etiqueta`, pero no incluye `id`, `sentimiento` ni `fecha`.

Los principales puntos de atención para una fase posterior son la alta cantidad de registros duplicados completos y la ausencia de una columna temporal. También debe decidirse si la columna `etiqueta` se mantendrá con ese nombre o si será documentada como equivalente funcional de `sentimiento` en el diseño del proyecto.

En esta fase no se realizó modelado, entrenamiento, limpieza avanzada, persistencia, implementación de API ni configuración de infraestructura.